# Generative Adversarial Network (DCGAN) pada CIFAR-10

**Buku:** *Deep Learning dengan Python* — Bab 14, Deep Learning Generatif

Generator dan diskriminator dilatih bergantian untuk membangkitkan gambar katak 32×32 dari CIFAR-10.

> Diadaptasi dari notebook pendamping *Deep Learning with Python* (F. Chollet, Manning; kode berlisensi MIT) dan diperbarui ke **Keras 3** untuk buku *Deep Learning dengan Python: Konsep, Kode, dan Penerapan Modern* (B. S. Negara).

**Lingkungan:** Google Colab (GPU disarankan), Python 3.10+, Keras 3. Notebook ini **membutuhkan backend TensorFlow** karena memakai `tf.GradientTape` untuk gradien terhadap masukan.

**Perubahan utama dari versi asli (Keras 2.0.8):**
- Trik `discriminator.trainable = False` sebelum `compile` model gabungan (perilakunya berubah di Keras 3) → subclass `keras.Model` dengan `train_step` dan dua optimizer.
- **Bug asli:** data diskalakan ke [0, 1] padahal generator berakhir dengan `tanh` ([-1, 1]). Di sini data diskalakan ke [-1, 1].
- `RMSprop(lr=..., decay=...)` → `RMSprop(learning_rate=...)` (argumen `decay` dihapus).
- Loop manual `train_on_batch` → `fit` pada `tf.data` dengan callback yang menyimpan sampel setiap epoch.
- Konvensi label buku tetap dipakai: 1 = palsu, 0 = asli, dengan derau kecil pada label.

In [ ]:
import os
# Backend wajib TensorFlow untuk notebook ini
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
print("Keras", keras.__version__, "| backend:", keras.backend.backend())
import tensorflow as tf
import numpy as np
from keras import layers

## 1. Data: semua gambar katak (kelas 6) dari CIFAR-10

In [ ]:
(x_latih, y_latih), _ = keras.datasets.cifar10.load_data()
katak = x_latih[y_latih.flatten() == 6].astype("float32")
katak = (katak - 127.5) / 127.5            # [-1, 1], sesuai tanh di generator
print(katak.shape)

UKURAN_BATCH = 64
dataset = tf.data.Dataset.from_tensor_slices(katak).shuffle(5000).batch(UKURAN_BATCH, drop_remainder=True)

## 2. Generator dan diskriminator (arsitektur buku)

In [ ]:
DIM_LATEN = 32

generator = keras.Sequential([
    keras.Input(shape=(DIM_LATEN,)),
    layers.Dense(128 * 16 * 16), layers.LeakyReLU(0.2),
    layers.Reshape((16, 16, 128)),
    layers.Conv2D(256, 5, padding="same"), layers.LeakyReLU(0.2),
    layers.Conv2DTranspose(256, 4, strides=2, padding="same"), layers.LeakyReLU(0.2),   # 16 -> 32
    layers.Conv2D(256, 5, padding="same"), layers.LeakyReLU(0.2),
    layers.Conv2D(256, 5, padding="same"), layers.LeakyReLU(0.2),
    layers.Conv2D(3, 7, activation="tanh", padding="same"),
], name="generator")

diskriminator = keras.Sequential([
    keras.Input(shape=(32, 32, 3)),
    layers.Conv2D(128, 3), layers.LeakyReLU(0.2),
    layers.Conv2D(128, 4, strides=2), layers.LeakyReLU(0.2),
    layers.Conv2D(128, 4, strides=2), layers.LeakyReLU(0.2),
    layers.Conv2D(128, 4, strides=2), layers.LeakyReLU(0.2),
    layers.Flatten(), layers.Dropout(0.4),
    layers.Dense(1, activation="sigmoid"),
], name="diskriminator")
generator.summary(); diskriminator.summary()

## 3. Model GAN dengan `train_step` kustom

In [ ]:
class GAN(keras.Model):
    def __init__(self, diskriminator, generator, dim_laten):
        super().__init__()
        self.diskriminator, self.generator, self.dim_laten = diskriminator, generator, dim_laten

    def compile(self, opt_d, opt_g, fungsi_loss):
        super().compile()
        self.opt_d, self.opt_g, self.fungsi_loss = opt_d, opt_g, fungsi_loss
        self.metrik_d = keras.metrics.Mean(name="loss_d")
        self.metrik_g = keras.metrics.Mean(name="loss_g")

    @property
    def metrics(self):
        return [self.metrik_d, self.metrik_g]

    def train_step(self, gambar_asli):
        n = tf.shape(gambar_asli)[0]

        # (a) Latih diskriminator: gambar palsu berlabel 1, asli berlabel 0
        z = tf.random.normal((n, self.dim_laten))
        gabungan = tf.concat([self.generator(z), gambar_asli], axis=0)
        label = tf.concat([tf.ones((n, 1)), tf.zeros((n, 1))], axis=0)
        label += 0.05 * tf.random.uniform(tf.shape(label))           # derau label
        with tf.GradientTape() as tape:
            loss_d = self.fungsi_loss(label, self.diskriminator(gabungan))
        grad = tape.gradient(loss_d, self.diskriminator.trainable_weights)
        self.opt_d.apply_gradients(zip(grad, self.diskriminator.trainable_weights))

        # (b) Latih generator: "bohongi" diskriminator agar menilai palsu sebagai asli (0)
        z = tf.random.normal((n, self.dim_laten))
        label_menyesatkan = tf.zeros((n, 1))
        with tf.GradientTape() as tape:
            loss_g = self.fungsi_loss(label_menyesatkan, self.diskriminator(self.generator(z)))
        grad = tape.gradient(loss_g, self.generator.trainable_weights)    # hanya bobot generator
        self.opt_g.apply_gradients(zip(grad, self.generator.trainable_weights))

        self.metrik_d.update_state(loss_d)
        self.metrik_g.update_state(loss_g)
        return {m.name: m.result() for m in self.metrics}

## 4. Callback penyimpan sampel

In [ ]:
import os
os.makedirs("gan_gambar", exist_ok=True)

class SimpanSampel(keras.callbacks.Callback):
    def __init__(self, n=8):
        super().__init__()
        self.z_tetap = tf.random.normal((n, DIM_LATEN))       # z tetap: progres mudah dibandingkan

    def on_epoch_end(self, epoch, logs=None):
        gambar = self.model.generator(self.z_tetap, training=False).numpy()
        gambar = np.clip(gambar * 127.5 + 127.5, 0, 255).astype("uint8")
        keras.utils.save_img(f"gan_gambar/epoch_{epoch + 1:03d}.png", np.concatenate(list(gambar), axis=1))

## 5. Latih
GAN sensitif terhadap hiperparameter. Pantau `loss_d`: bila mendekati 0, diskriminator menang telak dan generator berhenti belajar.

In [ ]:
gan = GAN(diskriminator, generator, DIM_LATEN)
gan.compile(opt_d=keras.optimizers.RMSprop(learning_rate=8e-4, clipvalue=1.0),
            opt_g=keras.optimizers.RMSprop(learning_rate=4e-4, clipvalue=1.0),
            fungsi_loss=keras.losses.BinaryCrossentropy())
riwayat = gan.fit(dataset, epochs=50, callbacks=[SimpanSampel()])

In [ ]:
import matplotlib.pyplot as plt

h = riwayat.history
plt.plot(h["loss_d"], label="Diskriminator"); plt.plot(h["loss_g"], label="Generator")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.show()

z = tf.random.normal((16, DIM_LATEN))
gambar = np.clip(generator(z, training=False).numpy() * 127.5 + 127.5, 0, 255).astype("uint8")
fig, ax = plt.subplots(2, 8, figsize=(14, 4))
for a, g in zip(ax.flat, gambar):
    a.imshow(g); a.axis("off")
plt.suptitle("Katak buatan generator"); plt.show()
generator.save("generator_katak.keras")